# การเตรียมสภาพแวดล้อมบน Google Colab

 ติดตั้ง Python Dependencies

In [ ]:
!pip -q install mysql-connector-python pandas
import mysql.connector
import pandas as pd
print(mysql.connector.__version__)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.7/21.7 MB 50.0 MB/s eta 0:00:00
26.7.0


ตรวจสอบ MySQL Server

In [ ]:
import mysql.connector
from mysql.connector import Error

In [ ]:
SERVER_CONFIG = {
 'host': '127.0.0.1',
 'port': 3306,
 'user': 'Medicine',
 'password': '123456',
 'connection_timeout': 10
}

In [ ]:
# 1. ติดตั้งและสตาร์ท MySQL Server บน Colab
!apt-get update > /dev/null
!apt-get install mysql-server > /dev/null
!service mysql start

# 2. สร้าง User 'Medicine' พร้อมตั้งรหัสผ่าน '123456'
!mysql -e "CREATE USER 'Medicine'@'localhost' IDENTIFIED WITH caching_sha2_password BY '123456';"

# 3. มอบสิทธิ์ให้ User 'Medicine' สามารถสร้างและจัดการฐานข้อมูลได้
!mysql -e "GRANT ALL PRIVILEGES ON *.* TO 'Medicine'@'localhost';"
!mysql -e "FLUSH PRIVILEGES;"

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
 * Starting MySQL database server mysqld
su: warning: cannot change directory to /nonexistent: No such file or directory
   ...done.


In [ ]:
# Step 1: connect ระดับ server
conn = mysql.connector.connect(**SERVER_CONFIG)
cur = conn.cursor()
cur.execute("CREATE DATABASE IF NOT EXISTS fake_news_db CHARACTER SET utf8mb4")
cur.close();
conn.close()

In [ ]:
# Step 2: config ที่ระบุ database
DB_CONFIG = {
 **SERVER_CONFIG,
 'database': 'fake_news_db',
 'autocommit': False
}

In [ ]:
def get_connection():
 return mysql.connector.connect(**DB_CONFIG)

สร้าง Schema สำ หรับ Fake News Mini Project

In [ ]:
def setup_database():
    conn = get_connection()
    cur = conn.cursor()
    try:
        cur.execute(
            """
      CREATE TABLE IF NOT EXISTS users (
      user_id VARCHAR(20) PRIMARY KEY,
      name VARCHAR(100) NOT NULL,
      email VARCHAR(100) UNIQUE,
      followers INT NOT NULL DEFAULT 0,
      post_count INT NOT NULL DEFAULT 0,
      is_verified BOOLEAN NOT NULL DEFAULT FALSE,
      created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP,
      CHECK (followers >= 0),
      CHECK (post_count >= 0)
      ) ENGINE=InnoDB
    """
        )
        cur.execute(
            """
    CREATE TABLE IF NOT EXISTS posts (
    post_id VARCHAR(20) PRIMARY KEY,
    author_id VARCHAR(20) NOT NULL,
    content TEXT NOT NULL,
    share_count INT NOT NULL DEFAULT 0,
    report_count INT NOT NULL DEFAULT 0,
    is_flagged BOOLEAN NOT NULL DEFAULT FALSE,
    created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP,
    CONSTRAINT fk_posts_author
    FOREIGN KEY (author_id) REFERENCES users(user_id)
    ON DELETE CASCADE,
    CHECK (share_count >= 0),
    CHECK (report_count >= 0)
    ) ENGINE=InnoDB
    """
        )
        cur.execute(
            """
    CREATE TABLE IF NOT EXISTS shares (
    share_id BIGINT AUTO_INCREMENT PRIMARY KEY,
    post_id VARCHAR(20) NOT NULL,
    user_id VARCHAR(20) NOT NULL,
    shared_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP,
    CONSTRAINT fk_shares_post FOREIGN KEY(post_id) REFERENCES posts(post_id) ON DELETE
    CASCADE,
    CONSTRAINT fk_shares_user FOREIGN KEY(user_id) REFERENCES users(user_id) ON DELETE
    CASCADE,
    CONSTRAINT uq_share UNIQUE(post_id, user_id)
    ) ENGINE=InnoDB
    """
        )
        conn.commit()
    except Error:
        conn.rollback()
        raise
    finally:
        cur.close()
        conn.close()


# เรียกใช้งานฟังก์ชันเพื่อสร้างตารางจริง
setup_database()

ใส่ข้อมูลตัวอย่างแบบทำซ้ำได้ (Idempotent-ish Reset)


In [ ]:
def insert_sample_data():
  conn = get_connection();
  cur = conn.cursor()
  try:
    cur.execute("DELETE FROM shares")
    cur.execute("DELETE FROM posts")
    cur.execute("DELETE FROM users")
    users = [
    ('U001','Alice','alice@test.com',1500,0,True),
    ('U002','Bob','bob@test.com',300,0,False),
    ('U003','Charlie','charlie@test.com',800,0,True),
    ('U004','David','david@test.com',120,0,False),
    ]
    cur.executemany("""
    INSERT INTO users(user_id,name,email,followers,post_count,is_verified)
    VALUES (%s,%s,%s,%s,%s,%s)
    """, users)
    posts = [
    ('P001','U001','Breaking: New AI model released!',0,0,False),
    ('P002','U002','Is this AI news fake?',0,0,False),
    ('P003','U003','Government announces new policy',0,0,False),
    ]
    cur.executemany("""
    INSERT INTO posts(post_id,author_id,content,share_count,report_count,is_flagged)
    VALUES (%s,%s,%s,%s,%s,%s)
    """, posts)
    conn.commit()
  except Error:
    conn.rollback(); raise
  finally:
    cur.close(); conn.close()
insert_sample_data()

# การจัดการ Transaction ใน MySQL ผ่าน Python

Create Post + Update Metric

In [ ]:
import uuid
def create_post_with_metrics(user_id, content):
  post_id = f"P{uuid.uuid4().hex[:8]}"
  conn = get_connection(); cur = conn.cursor()
  try:
    cur.execute("""
    INSERT INTO posts(post_id,author_id,content,share_count,report_count)
    VALUES (%s,%s,%s,0,0)
    """, (post_id,user_id,content))
    cur.execute("""
    UPDATE users SET post_count = post_count + 1
    WHERE user_id = %s
    """, (user_id,))
    if cur.rowcount != 1:
      raise ValueError(f"Unknown user_id: {user_id}")
    conn.commit()
    return post_id
  except Exception:
    conn.rollback()
    raise
  finally:
    cur.close(); conn.close()

Deadlock Retry

In [ ]:
import time, random
from mysql.connector import Error
DEADLOCK = 1213
LOCK_WAIT_TIMEOUT = 1205
def run_with_retry(fn, max_retries=5):
  for attempt in range(max_retries):
    try:
      return fn()
    except Error as e:
      if e.errno not in (DEADLOCK, LOCK_WAIT_TIMEOUT) or attempt == max_retries-1:
        raise
    # exponential backoff + jitter
    wait = 0.1 * (2 ** attempt) + random.uniform(0, 0.05)
    time.sleep(wait)

Unsafe Read-Modify-Write: Lost Update

In [ ]:
import concurrent.futures, time
def bot_share_unsafe(bot_id, post_id):
  conn = get_connection(); cur = conn.cursor()
  try:
    cur.execute("SELECT share_count FROM posts WHERE post_id=%s", (post_id,))
    current = cur.fetchone()[0]
    time.sleep(0.02) # เพิ่มโอ พิ่ กาส race
    cur.execute("UPDATE posts SET share_count=%s WHERE post_id=%s",
    (current+1, post_id))
    conn.commit()
    return current + 1
  except Exception:
    conn.rollback(); raise
  finally:
    cur.close(); conn.close()


แก้ด้วย Atomic UPDATE

In [ ]:
def bot_share_atomic(bot_id, post_id):
  conn = get_connection(); cur = conn.cursor()
  try:
    cur.execute("""
    UPDATE posts
    SET share_count = share_count + 1
    WHERE post_id = %s
    """, (post_id,))
    conn.commit()
  except Exception:
    conn.rollback(); raise
  finally:
    cur.close(); conn.close()


จำลอง Deadlock จริงด้วย Row Locks

In [ ]:
def transfer_followers(from_user, to_user, amount):
  conn = get_connection();
  cur = conn.cursor()
  try:
    cur.execute("UPDATE users SET followers=followers-%s WHERE user_id=%s",
    (amount, from_user))
    time.sleep(0.05)
    cur.execute("UPDATE users SET followers=followers+%s WHERE user_id=%s",
    (amount, to_user))
    conn.commit()
  except Error:
    conn.rollback(); raise
  finally:
    cur.close(); conn.close()
# Thread 1: U001 -> U002
# Thread 2: U002 -> U001
# ลำ ดับ lock ตรงข้ามกันอาจเกิด deadlock; จับ error 1213 แล้ว retry ทั้ง transaction

ทดสอบ Isolation Level อย่างปลอดภัย

In [ ]:
import threading, time
def reader(level, ready, writer_done, result):
  conn=get_connection(); cur=conn.cursor()
  try:
    cur.execute(f"SET SESSION TRANSACTION ISOLATION LEVEL {level}")
    cur.execute("START TRANSACTION")
    cur.execute("SELECT share_count FROM posts WHERE post_id='P001'")
    result['first'] = cur.fetchone()[0]
    ready.set()
    writer_done.wait(timeout=8)
    cur.execute("SELECT share_count FROM posts WHERE post_id='P001'")
    result['second'] = cur.fetchone()[0]
    conn.commit()
  finally:
    cur.close(); conn.close()

def writer(ready, writer_done, result):
  ready.wait(timeout=8)
  conn=get_connection(); cur=conn.cursor()
  try:
    cur.execute("SET SESSION innodb_lock_wait_timeout=3")
    cur.execute("UPDATE posts SET share_count=999 WHERE post_id='P001'")
    conn.commit(); result['writer']='committed'
  except Error as e:
    conn.rollback(); result['writer']=f"error {e.errno}"
  finally:
    cur.close(); conn.close(); writer_done.set()

def run_isolation_test(level):
  ready=threading.Event(); done=threading.Event(); result={}
  t1=threading.Thread(target=reader,args=(level,ready,done,result))
  t2=threading.Thread(target=writer,args=(ready,done,result))
  t1.start(); t2.start(); t1.join(); t2.join()
  return result

# ตรวจสอบ Redo Log และ Recovery ใน MySQL 8.4
ดูInnoDB Status

In [ ]:
conn = get_connection(); cur = conn.cursor()
cur.execute("SHOW ENGINE INNODB STATUS")
status = cur.fetchone()[2]
print(status)
cur.close(); conn.close()


2026-09-03 07:33:57 132785833473600 INNODB MONITOR OUTPUT
Per second averages calculated from the last 43 seconds
-----------------
BACKGROUND THREAD
-----------------
srv_master_thread loops: 3 srv_active, 0 srv_shutdown, 370 srv_idle
srv_master_thread log flush and writes: 0
----------
SEMAPHORES
----------
OS WAIT ARRAY INFO: reservation count 177
OS WAIT ARRAY INFO: signal count 169
RW-shared spins 0, rounds 0, OS waits 0
RW-excl spins 0, rounds 0, OS waits 0
RW-sx spins 0, rounds 0, OS waits 0
Spin rounds per wait: 0.00 RW-shared, 0.00 RW-excl, 0.00 RW-sx
------------------------
LATEST FOREIGN KEY ERROR
------------------------
2026-09-03 07:27:45 132785833473600 Transaction:
TRANSACTION 2346, ACTIVE 0 sec inserting
mysql tables in use 1, locked 1
3 lock struct(s), heap size 1128, 1 row lock(s), undo log entries 1
MySQL thread id 17, OS thread handle 132785833473600, query id 43 localhost 127.0.0.1 Medicine update
INSERT INTO posts(post_id,author_id,content,share_count)
  VALUES

สาธิต Redo Generation

In [ ]:
def get_redo_bytes():
 conn=get_connection(); cur=conn.cursor()
 cur.execute("SHOW GLOBAL STATUS LIKE 'Innodb_os_log_written'")
 value=int(cur.fetchone()[1])
 cur.close(); conn.close()
 return value
before = get_redo_bytes()
conn=get_connection(); cur=conn.cursor()
try:
 for i in range(1000):
  cur.execute("""
  INSERT INTO posts(post_id,author_id,content,share_count)
  VALUES (%s,'U001',%s,0)
  """, (f'BULK_{i:04d}', f'Bulk insert test {i}'))
 conn.commit()
finally:
 cur.close(); conn.close()

after = get_redo_bytes()
print('Redo bytes generated (approx.):', after-before)


Redo bytes generated (approx.): 350208


# Lab 1: วิเคราะห์ Dirty Read

In [ ]:
def test_dirty_read(isolation_level):
    print(f"\n=== ทดสอบด้วย ISOLATION LEVEL: {isolation_level} ===")

    # Connection 1 (T1) - จำลอง Session ที่ทำการแก้ไขข้อมูล
    conn1 = get_connection()
    cur1 = conn1.cursor()

    # Connection 2 (T2) - จำลอง Session ที่เข้ามาอ่านข้อมูล
    conn2 = get_connection()
    cur2 = conn2.cursor()

    try:
        # T1: เริ่ม Transaction และ Update ข้อมูล (แต่ยังไม่ COMMIT)
        cur1.execute("START TRANSACTION")
        cur1.execute("UPDATE posts SET share_count=777 WHERE post_id='P001'")
        print("T1: สั่ง UPDATE posts เปลี่ยน share_count=777 (ยังไม่ COMMIT)")

        # T2: กำหนด Isolation level แล้วลอง SELECT ค่าออกมาดู
        cur2.execute(f"SET SESSION TRANSACTION ISOLATION LEVEL {isolation_level}")
        cur2.execute("SELECT share_count FROM posts WHERE post_id='P001'")
        result_before = cur2.fetchone()[0]
        print(f"T2: ลองอ่านค่า share_count ได้ผลลัพธ์ = {result_before}")

        # T1: เกิด Error หรือเปลี่ยนใจ สั่ง ROLLBACK ยกเลิกการอัปเดต
        conn1.rollback()
        print("T1: สั่ง ROLLBACK ข้อมูล")

        # T2: ลองอ่านค่าอีกครั้งเพื่อดูผลลัพธ์ที่แท้จริง
        cur2.execute("SELECT share_count FROM posts WHERE post_id='P001'")
        result_after = cur2.fetchone()[0]
        print(f"T2: ลองอ่านค่าอีกครั้งหลัง T1 Rollback ได้ผลลัพธ์ = {result_after}")

    finally:
        cur1.close()
        conn1.close()
        cur2.close()
        conn2.close()

# ทดสอบเปรียบเทียบ 2 รูปแบบ
test_dirty_read("READ UNCOMMITTED")
test_dirty_read("READ COMMITTED")


=== ทดสอบด้วย ISOLATION LEVEL: READ UNCOMMITTED ===
T1: สั่ง UPDATE posts เปลี่ยน share_count=777 (ยังไม่ COMMIT)
T2: ลองอ่านค่า share_count ได้ผลลัพธ์ = 777
T1: สั่ง ROLLBACK ข้อมูล
T2: ลองอ่านค่าอีกครั้งหลัง T1 Rollback ได้ผลลัพธ์ = 0

=== ทดสอบด้วย ISOLATION LEVEL: READ COMMITTED ===
T1: สั่ง UPDATE posts เปลี่ยน share_count=777 (ยังไม่ COMMIT)
T2: ลองอ่านค่า share_count ได้ผลลัพธ์ = 0
T1: สั่ง ROLLBACK ข้อมูล
T2: ลองอ่านค่าอีกครั้งหลัง T1 Rollback ได้ผลลัพธ์ = 0


สรุปผลลัพธ์เชิงวิเคราะห์ที่ได้จาก Lab 1 (นำไปใช้ตอบท้ายแล็บได้)

READ UNCOMMITTED: จะเห็นค่า 777 ทะลุออกมาทั้งที่ T1 ยังไม่ได้ Commit ซึ่งเป็นความเสี่ยงต่อ Business Logic อย่างมาก เพราะถ้าระบบนำค่า 777 นี้ไปคำนวณต่อ แต่สุดท้าย T1 สั่ง Rollback ข้อมูลในระบบจะผิดพลาดทันที

READ COMMITTED: จะเห็นเป็นค่าเริ่มต้นที่ถูกต้อง (เช่น 0) เพราะระบบจะยอมให้อ่านเฉพาะข้อมูลที่ได้รับการ Commit เรียบร้อยแล้วเท่านั้น

# Lab 2: Atomic Transaction สำ หรับ Report Fake News

สร้างตาราง reports

In [ ]:
def setup_reports_table():
    conn = get_connection()
    cur = conn.cursor()
    try:
        cur.execute("""
            CREATE TABLE IF NOT EXISTS reports (
                report_id INT AUTO_INCREMENT PRIMARY KEY,
                post_id VARCHAR(20) NOT NULL,
                user_id VARCHAR(20) NOT NULL,
                reason TEXT,
                reported_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP,
                CONSTRAINT fk_reports_post FOREIGN KEY (post_id) REFERENCES posts(post_id) ON DELETE CASCADE,
                CONSTRAINT fk_reports_user FOREIGN KEY (user_id) REFERENCES users(user_id) ON DELETE CASCADE,
                CONSTRAINT uq_report UNIQUE (post_id, user_id)
            ) ENGINE=InnoDB
        """)
        conn.commit()
        print("สร้างตาราง reports สำเร็จ")
    except Exception as e:
        conn.rollback()
        print(f"Error: {e}")
    finally:
        cur.close()
        conn.close()

setup_reports_table()

สร้างตาราง reports สำเร็จ


สร้างฟังก์ชัน report_fake_news (หัวใจหลักของ Lab)

In [ ]:
from mysql.connector import Error

def report_fake_news(user_id, post_id, reason):
    conn = get_connection()
    cur = conn.cursor()
    threshold = 3 # กำหนด Threshold ไว้ที่ 3 reports ให้แบนโพสต์

    try:
        # 1. INSERT รายงานลง reports (ถ้าซ้ำจะเกิด Error จาก UNIQUE constraint ทันที)
        cur.execute("""
            INSERT INTO reports (post_id, user_id, reason)
            VALUES (%s, %s, %s)
        """, (post_id, user_id, reason))

        # 2. Atomic UPDATE เพิ่มยอด report_count ขึ้น 1
        cur.execute("""
            UPDATE posts
            SET report_count = report_count + 1
            WHERE post_id = %s
        """, (post_id,))

        # 3. เช็คเงื่อนไขถ้า report_count ถึงเกณฑ์ให้ปักธง is_flagged = TRUE
        cur.execute("""
            UPDATE posts
            SET is_flagged = TRUE
            WHERE post_id = %s AND report_count >= %s
        """, (post_id, threshold))

        conn.commit()
        print(f"User '{user_id}' กด Report โพสต์ '{post_id}' สำเร็จ")

    except Error as e:
        # หากเกิด Error เช่น report ซ้ำ (Duplicate Key) หรือ Foreign Key ผิด จะทำการ Rollback ทั้งหมด
        conn.rollback()
        print(f"User '{user_id}' Report ล้มเหลว (Rollback): {e.msg}")
    finally:
        cur.close()
        conn.close()

จำลองการทำงานพร้อมกัน (Concurrency Test)

In [ ]:
import concurrent.futures

def test_concurrent_reports():
    print("\n=== เริ่มทดสอบการ Report โพสต์ P001 ===")

    # รีเซ็ตข้อมูลให้ P001 ก่อนเริ่มทดสอบ
    conn = get_connection()
    cur = conn.cursor()
    cur.execute("DELETE FROM reports")
    cur.execute("UPDATE posts SET report_count = 0, is_flagged = FALSE WHERE post_id = 'P001'")
    conn.commit()
    cur.close(); conn.close()

    # จำลอง User 4 คนเข้ามากด Report (สังเกตว่า 'U001' จะกดซ้ำ 2 ครั้ง)
    test_users = ['U001', 'U002', 'U003', 'U004', 'U001']

    # สั่งให้ทำงานพร้อมกัน
    with concurrent.futures.ThreadPoolExecutor(max_workers=5) as executor:
        futures = [executor.submit(report_fake_news, user, 'P001', 'เนื้อหาไม่เป็นความจริง') for user in test_users]
        concurrent.futures.wait(futures)

    # ตรวจสอบผลลัพธ์สุดท้าย
    conn = get_connection()
    cur = conn.cursor(dictionary=True)
    cur.execute("SELECT report_count, is_flagged FROM posts WHERE post_id = 'P001'")
    post_status = cur.fetchone()
    print("\n=== ผลลัพธ์สุดท้ายของโพสต์ P001 ===")
    print(post_status)
    cur.close(); conn.close()

test_concurrent_reports()


=== เริ่มทดสอบการ Report โพสต์ P001 ===
User 'U002' กด Report โพสต์ 'P001' สำเร็จ
User 'U004' Report ล้มเหลว (Rollback): Deadlock found when trying to get lock; try restarting transaction
User 'U003' Report ล้มเหลว (Rollback): Deadlock found when trying to get lock; try restarting transaction
User 'U001' Report ล้มเหลว (Rollback): Deadlock found when trying to get lock; try restarting transaction
User 'U001' กด Report โพสต์ 'P001' สำเร็จ

=== ผลลัพธ์สุดท้ายของโพสต์ P001 ===
{'report_count': 2, 'is_flagged': 0}


การใช้ Atomic UPDATE (report_count = report_count + 1)ในภาษา SQL โดยตรง จะช่วยให้ Database Engine เป็นคนจัดการเรื่อง Record Locking โดยอัตโนมัติ ทำให้หมดปัญหา "Lost Update" จากการดึงค่ามาบวกเองในแอปพลิเคชัน (Read-Modify-Write) เมื่อมีคนกด Report พร้อมกันหลายร้อยคน

นอกจากนี้ การมัดรวม INSERT และ UPDATE ให้อยู่ใน Transaction เดียวกัน เมื่อมีคนพยายามกด Report ซ้ำ (ละเมิด UNIQUE Constraint)ระบบจะเด้ง Error ไปเข้า Except และดึงคำสั่ง conn.rollback()ทำให้กระบวนการอัปเดตยอดที่ค้างอยู่ถูกตีตกไปทั้งชุด ยอด Report จึงไม่เพิ่มขึ้นหากการแจ้งนั้นไม่ถูกต้องครับ

In [ ]:
import time
import uuid
import statistics

# สร้างตารางจำลองสำหรับการทำ Benchmark
def setup_benchmark_table():
    conn = get_connection()
    cur = conn.cursor()
    cur.execute("""
        CREATE TABLE IF NOT EXISTS benchmark_test (
            id VARCHAR(50) PRIMARY KEY,
            val INT
        ) ENGINE=InnoDB
    """)
    conn.commit()
    cur.close(); conn.close()

# Case A: Autocommit = True (Commit ทุกคำสั่ง INSERT)
def benchmark_autocommit(n_rows):
    conn = get_connection()
    conn.autocommit = True
    cur = conn.cursor()

    cur.execute("TRUNCATE TABLE benchmark_test") # เคลียร์ข้อมูลก่อนรัน

    start_time = time.time()
    for i in range(n_rows):
        cur.execute("INSERT INTO benchmark_test (id, val) VALUES (%s, %s)", (uuid.uuid4().hex, i))

    elapsed = time.time() - start_time
    cur.close(); conn.close()
    return elapsed

# Case B: Autocommit = False (Explicit Transaction - Commit ครั้งเดียวตอนจบ)
def benchmark_explicit(n_rows):
    conn = get_connection()
    conn.autocommit = False
    cur = conn.cursor()

    cur.execute("TRUNCATE TABLE benchmark_test")

    start_time = time.time()
    try:
        for i in range(n_rows):
            cur.execute("INSERT INTO benchmark_test (id, val) VALUES (%s, %s)", (uuid.uuid4().hex, i))
        conn.commit() # Commit ครั้งเดียวรวด
    except Exception:
        conn.rollback()
        raise

    elapsed = time.time() - start_time
    cur.close(); conn.close()
    return elapsed

def run_benchmark():
    setup_benchmark_table()
    n_rows = 1000 # ทดสอบกับ 1,000 แถว
    runs = 5 # เก็บข้อมูล 5 รอบ

    print(f"=== เริ่มการทดสอบ Benchmark (INSERT {n_rows} แถว จำนวน {runs} รอบ) ===")

    # Warm up database (รันวอร์มก่อนเก็บค่าจริง)
    benchmark_autocommit(100)

    auto_times = [benchmark_autocommit(n_rows) for _ in range(runs)]
    explicit_times = [benchmark_explicit(n_rows) for _ in range(runs)]

    print(f"Case A (Autocommit): ค่าเฉลี่ย (Mean) = {statistics.mean(auto_times):.4f} วินาที")
    print(f"Case B (Explicit Transaction): ค่าเฉลี่ย (Mean) = {statistics.mean(explicit_times):.4f} วินาที")

run_benchmark()

=== เริ่มการทดสอบ Benchmark (INSERT 1000 แถว จำนวน 5 รอบ) ===
Case A (Autocommit): ค่าเฉลี่ย (Mean) = 7.0238 วินาที
Case B (Explicit Transaction): ค่าเฉลี่ย (Mean) = 0.4728 วินาที


ทำไม Autocommit จึงช้ากว่ามาก? การที่ Autocommit ทำงานในทุกๆ Statement จะไปเพิ่ม Overhead ในระดับ OS (การทำ Log flush และ fsync ลงดิสก์) โดยเฉพาะถ้าเซิร์ฟเวอร์ตั้งค่า innodb_flush_log_at_trx_commit=1 ที่บังคับให้ต้องเขียน Redo Log ลงดิสก์ทันทีหลังจบ Transaction เพื่อป้องกันข้อมูลหาย

ข้อควรระวังของ Explicit Transaction ขนาดใหญ่: แม้การรวบทำ Explicit Transaction จะเร็วกว่า แต่หากรวม INSERT ไว้เยอะเกินไปใน Transaction เดียว (เช่น หลายล้านแถว) จะทำให้เกิดต้นทุนในการจัดการ Undo Log ที่บวมขึ้น, การกินระยะเวลาถือ Lock นานขึ้น (Lock Duration), และส่งผลให้ระยะเวลาการทำ Recovery ของ Database นานขึ้นเมื่อเกิดระบบล่ม (Crash)

Deadlock Detection และ Retry

In [ ]:
import threading
import time
import random
from mysql.connector import Error

DEADLOCK = 1213
LOCK_WAIT_TIMEOUT = 1205

# ฟังก์ชันจัดการ Retry ทั้งชุด Transaction เมื่อเจอ Deadlock
def run_with_retry(fn, name, first_user, second_user, max_retries=5):
    for attempt in range(max_retries):
        try:
            return fn(name, first_user, second_user)
        except Error as e:
            # ถ้าเป็น Error ปกติ หรือ Retry จนหมดโควต้าแล้ว ให้พ่น Error ออกมา
            if e.errno not in (DEADLOCK, LOCK_WAIT_TIMEOUT) or attempt == max_retries - 1:
                raise

            # การทำ Exponential Backoff + Jitter ตามสูตรในเอกสาร
            wait = 0.1 * (2 ** attempt) + random.uniform(0, 0.05)
            print(f"[{name}] เกิด Deadlock! (Attempt {attempt+1}) กำลังรอ {wait:.2f} วินาทีเพื่อเริ่ม Transaction ใหม่...")
            time.sleep(wait)

# ฟังก์ชันหลักที่เป็น Business Logic (ต้องสำเร็จทั้งชุด หรือ Rollback ทั้งชุด)
def update_followers(name, first_user, second_user):
    conn = get_connection()
    cur = conn.cursor()
    try:
        cur.execute("START TRANSACTION")
        print(f"[{name}] กำลังล็อก {first_user}")
        cur.execute("UPDATE users SET followers = followers + 1 WHERE user_id = %s", (first_user,))

        # ถ่วงเวลา 1 วินาที เพื่อการันตีว่าอีก Thread จะเข้าล็อกอีก Row แน่นอน
        time.sleep(1)

        print(f"[{name}] พยายามล็อก {second_user} (อาจเกิดการรอตรงนี้)")
        cur.execute("UPDATE users SET followers = followers + 1 WHERE user_id = %s", (second_user,))

        conn.commit()
        print(f"[{name}] อัปเดตสำเร็จเรียบร้อย (Committed)")
    except Error:
        conn.rollback() # Rollback ทุกอย่างใน Transaction นี้ทิ้ง
        raise
    finally:
        cur.close()
        conn.close()

def run_deadlock_test():
    print("=== เริ่มการจำลอง Deadlock ===")

    # Thread 1: ล็อก U001 แล้วไป U002
    t1 = threading.Thread(target=run_with_retry, args=(update_followers, "T1", "U001", "U002"))
    # Thread 2: ล็อก U002 แล้วไป U001 (ลำดับตรงข้ามกัน)
    t2 = threading.Thread(target=run_with_retry, args=(update_followers, "T2", "U002", "U001"))

    t1.start()
    t2.start()
    t1.join()
    t2.join()

# ตรวจสอบสถานะว่า Database เจอ Deadlock จริงไหม
def show_deadlock_status():
    conn = get_connection()
    cur = conn.cursor()
    cur.execute("SHOW ENGINE INNODB STATUS")
    status = cur.fetchone()[2]

    print("\n=== LATEST DETECTED DEADLOCK จาก INNODB STATUS (ตัดมาเฉพาะส่วนสำคัญ) ===")
    if "LATEST DETECTED DEADLOCK" in status:
        # ตัดข้อความมาแสดงเฉพาะส่วนที่เป็นข้อมูล Deadlock ล่าสุด
        info = status.split("LATEST DETECTED DEADLOCK")[1].split("WE ROLL BACK TRANSACTION")[0]
        print(info.strip())
    else:
        print("ไม่พบประวัติ Deadlock ใน Status")

    cur.close()
    conn.close()

# สั่งรันการจำลองและดูสถานะ
run_deadlock_test()
show_deadlock_status()

=== เริ่มการจำลอง Deadlock ===
[T1] กำลังล็อก U001
[T2] กำลังล็อก U002
[T2] พยายามล็อก U001 (อาจเกิดการรอตรงนี้)
[T1] พยายามล็อก U002 (อาจเกิดการรอตรงนี้)
[T1] เกิด Deadlock! (Attempt 1) กำลังรอ 0.15 วินาทีเพื่อเริ่ม Transaction ใหม่...
[T2] อัปเดตสำเร็จเรียบร้อย (Committed)
[T1] กำลังล็อก U001
[T1] พยายามล็อก U002 (อาจเกิดการรอตรงนี้)
[T1] อัปเดตสำเร็จเรียบร้อย (Committed)

=== LATEST DETECTED DEADLOCK จาก INNODB STATUS (ตัดมาเฉพาะส่วนสำคัญ) ===
------------------------
2026-09-03 08:46:01 132785974093376
*** (1) TRANSACTION:
TRANSACTION 7729, ACTIVE 1 sec starting index read
mysql tables in use 1, locked 1
LOCK WAIT 3 lock struct(s), heap size 1128, 2 row lock(s), undo log entries 1
MySQL thread id 47, OS thread handle 132785830303296, query id 11384 localhost 127.0.0.1 Medicine updating
UPDATE users SET followers = followers + 1 WHERE user_id = 'U001'

*** (1) HOLDS THE LOCK(S):
RECORD LOCKS space id 2 page no 4 n bits 72 index PRIMARY of table `fake_news_db`.`users` trx id 7729 loc

เมื่อเกิดความขัดแย้งที่ทั้งสองฝั่งต่างรอให้อีกฝ่ายปลดล็อก MySQL จะใช้ระบบ Deadlock Detection ตรวจพบปัญหาทันที และสุ่มเตะ Transaction ใด Transaction หนึ่งออก (Rollback Victim) ทำให้ฝั่งนั้นได้รับ Error 1213 (Deadlock found)  เราไม่สามารถสั่ง Retry แค่เฉพาะคำสั่ง UPDATE ตัวที่ล้มเหลวได้ เพราะเมื่อ InnoDB มองว่า Transaction นี้น่าจะไปต่อไม่ได้แล้ว มันจะ Rollback ทิ้งทั้งชุดแบบอัตโนมัติ ฝั่งแอปพลิเคชันจึงจำเป็นต้องทำ Retry ลูปครอบ Business Logic ไว้ทั้งหมดเสมอ  การใช้ Exponential Backoff + Jitter ช่วยกระจายระยะเวลาการรอ ทำให้ลดโอกาสที่ Thread ทั้งสองฝั่งจะตื่นขึ้นมายิงคำสั่งและเกิด Deadlock ชนกันซ้ำซากรอบที่สอง

# Lab Assignment: Transaction, Concurrency & Recovery

Cell 0: เตรียม Environment และข้อมูล (ต้องรันก่อนเริ่มข้อ A-E)

In [ ]:
# พัฒนาโดย นายปกานต์ วงษ์ท่าเรือ 664245056
!pip -q install mysql-connector-python pandas

import mysql.connector
from mysql.connector import Error
import concurrent.futures
import threading
import time
import random

SERVER_CONFIG = {'host': '127.0.0.1', 'port': 3306, 'user': 'Medicine', 'password': '123456', 'connection_timeout': 10}
DB_CONFIG = {**SERVER_CONFIG, 'database': 'fake_news_db', 'autocommit': False}

# สร้าง Database
conn = mysql.connector.connect(**SERVER_CONFIG)
cur = conn.cursor()
cur.execute("CREATE DATABASE IF NOT EXISTS fake_news_db CHARACTER SET utf8mb4")
cur.close(); conn.close()

def get_connection():
    return mysql.connector.connect(**DB_CONFIG)

def setup_all():
    conn = get_connection(); cur = conn.cursor()
    try:
        # สร้าง Schema
        cur.execute("CREATE TABLE IF NOT EXISTS users (user_id VARCHAR(20) PRIMARY KEY, name VARCHAR(100) NOT NULL, email VARCHAR(100) UNIQUE, followers INT NOT NULL DEFAULT 0, post_count INT NOT NULL DEFAULT 0, is_verified BOOLEAN NOT NULL DEFAULT FALSE, created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP)")
        cur.execute("CREATE TABLE IF NOT EXISTS posts (post_id VARCHAR(20) PRIMARY KEY, author_id VARCHAR(20) NOT NULL, content TEXT NOT NULL, share_count INT NOT NULL DEFAULT 0, report_count INT NOT NULL DEFAULT 0, is_flagged BOOLEAN NOT NULL DEFAULT FALSE, created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP, CONSTRAINT fk_posts_author FOREIGN KEY (author_id) REFERENCES users (user_id) ON DELETE CASCADE)")
        cur.execute("CREATE TABLE IF NOT EXISTS reports (report_id INT AUTO_INCREMENT PRIMARY KEY, post_id VARCHAR(20) NOT NULL, user_id VARCHAR(20) NOT NULL, UNIQUE(post_id, user_id))")

        # ล้างข้อมูลเก่าและเพิ่มข้อมูลตัวอย่าง
        cur.execute("DELETE FROM reports"); cur.execute("DELETE FROM posts"); cur.execute("DELETE FROM users")
        cur.executemany("INSERT INTO users (user_id, name, email) VALUES (%s, %s, %s)", [('U001', 'Alice', 'alice@test.com'), ('U002', 'Bob', 'bob@test.com')])
        cur.executemany("INSERT INTO posts (post_id, author_id, content) VALUES (%s, %s, %s)", [('P001', 'U001', 'Breaking News'), ('P002', 'U002', 'AI News')])
        conn.commit()
    except Error:
        conn.rollback(); raise
    finally:
        cur.close(); conn.close()

setup_all()
print("Setup Database Success")

Setup Database Success


ACID Transaction
สร้าง Business operation แบบ 3 Statements และพิสูจน์การ Rollback เมื่อเกิด Error

In [ ]:
def report_fake_news(user_id, post_id):
    conn = get_connection(); cur = conn.cursor()
    try:
        # 1. INSERT ข้อมูลรายงาน
        cur.execute("INSERT INTO reports (post_id, user_id) VALUES (%s, %s)", (post_id, user_id))
        # 2. UPDATE เพิ่มยอด
        cur.execute("UPDATE posts SET report_count = report_count + 1 WHERE post_id = %s", (post_id,))
        # 3. UPDATE แบนโพสต์หากถึงเกณฑ์
        cur.execute("UPDATE posts SET is_flagged = TRUE WHERE post_id = %s AND report_count >= 3", (post_id,))
        conn.commit()
        print(f"{user_id} reported {post_id} successfully")
    except Error as e:
        conn.rollback() # Rollback ทุก operation หากมี error (เช่น แจ้งซ้ำ)
        print(f"Rollback {user_id}: {e.msg}")
    finally:
        cur.close(); conn.close()

# ทดสอบรันให้เกิด Error จากการละเมิด UNIQUE Constraint
report_fake_news('U001', 'P001')
report_fake_news('U001', 'P001') # จะโดน Rollback

Lost Update
เปรียบเทียบการอัปเดตแบบ Read-Modify-Write ที่ทำให้เกิด Lost Update กับการใช้ Atomic UPDATE

In [ ]:
def share_unsafe(post_id):
    conn = get_connection(); cur = conn.cursor()
    try:
        cur.execute("SELECT share_count FROM posts WHERE post_id=%s", (post_id,))
        current = cur.fetchone()[0]
        time.sleep(0.05) # เพิ่มโอกาสเกิด Race Condition
        cur.execute("UPDATE posts SET share_count=%s WHERE post_id=%s", (current + 1, post_id))
        conn.commit()
    except Exception: conn.rollback()
    finally: cur.close(); conn.close()

def share_atomic(post_id):
    conn = get_connection(); cur = conn.cursor()
    try:
        # ใช้คำสั่งเดียวให้ InnoDB จัดการ Record locking อัตโนมัติ
        cur.execute("UPDATE posts SET share_count = share_count + 1 WHERE post_id=%s", (post_id,))
        conn.commit()
    except Exception: conn.rollback()
    finally: cur.close(); conn.close()

# รีเซ็ตค่าและทดสอบ Unsafe
conn = get_connection(); cur = conn.cursor()
cur.execute("UPDATE posts SET share_count=0 WHERE post_id='P001'"); conn.commit()
with concurrent.futures.ThreadPoolExecutor(max_workers=10) as ex:
    list(ex.map(lambda i: share_unsafe('P001'), range(10)))
cur.execute("SELECT share_count FROM posts WHERE post_id='P001'")
print(f"Unsafe Share Count (ควรเป็น 10 แต่เกิด Lost Update): {cur.fetchone()[0]}")

# รีเซ็ตค่าและทดสอบ Atomic
cur.execute("UPDATE posts SET share_count=0 WHERE post_id='P002'"); conn.commit()
with concurrent.futures.ThreadPoolExecutor(max_workers=10) as ex:
    list(ex.map(lambda i: share_atomic('P002'), range(10)))
cur.execute("SELECT share_count FROM posts WHERE post_id='P002'")
print(f"Atomic Share Count (ได้ 10 เสมอ): {cur.fetchone()[0]}")
cur.close(); conn.close()

Isolation
ทดสอบ Dirty Read ด้วยการเปรียบเทียบ READ UNCOMMITTED และ READ COMMITTED

In [ ]:
def test_dirty_read(isolation_level):
    conn1 = get_connection(); cur1 = conn1.cursor()
    conn2 = get_connection(); cur2 = conn2.cursor()
    try:
        # T1 สั่งแก้ไขข้อมูลแต่ยังไม่ Commit
        cur1.execute("START TRANSACTION")
        cur1.execute("UPDATE posts SET share_count=999 WHERE post_id='P001'")

        # T2 พยายามอ่านข้อมูล
        cur2.execute(f"SET SESSION TRANSACTION ISOLATION LEVEL {isolation_level}")
        cur2.execute("START TRANSACTION")
        cur2.execute("SELECT share_count FROM posts WHERE post_id='P001'")
        print(f"ผลจาก {isolation_level}: อ่านค่าได้ {cur2.fetchone()[0]}")

        conn1.rollback() # T1 ยกเลิกข้อมูลทิ้ง
    finally:
        cur1.close(); conn1.close(); cur2.close(); conn2.close()

test_dirty_read("READ UNCOMMITTED")
test_dirty_read("READ COMMITTED")

Deadlock
จำลอง Deadlock ด้วยการล็อกไขว้ลำดับ และทำ Retry ทั้งชุดพร้อม Exponential Backoff

In [ ]:
def run_with_retry(fn, user1, user2):
    for attempt in range(5):
        try:
            return fn(user1, user2)
        except Error as e:
            if e.errno not in (1213, 1205): raise
            print(f"Deadlock detected! Retrying attempt {attempt+1}")
            time.sleep(0.1 * (2 ** attempt) + random.uniform(0, 0.05)) # Exponential backoff + jitter[cite: 1]

def cross_update(user1, user2):
    conn = get_connection(); cur = conn.cursor()
    try:
        cur.execute("START TRANSACTION")
        cur.execute("UPDATE users SET followers = followers + 1 WHERE user_id = %s", (user1,))
        time.sleep(0.5) # รอให้อีก Thread ล็อกอีก Row[cite: 1]
        cur.execute("UPDATE users SET followers = followers + 1 WHERE user_id = %s", (user2,))
        conn.commit()
    except Error:
        conn.rollback(); raise
    finally:
        cur.close(); conn.close()

t1 = threading.Thread(target=run_with_retry, args=(cross_update, "U001", "U002"))
t2 = threading.Thread(target=run_with_retry, args=(cross_update, "U002", "U001"))
t1.start(); t2.start(); t1.join(); t2.join()
print("Deadlock retry mechanism finished")

Redo/Undo
ตรวจสอบปริมาณ Redo Log ที่เพิ่มขึ้นเพื่อพิสูจน์การทำงานของ WAL (Write-Ahead Logging)

In [ ]:
def get_redo_bytes():
    conn = get_connection(); cur = conn.cursor()
    cur.execute("SHOW GLOBAL STATUS LIKE 'Innodb_os_log_written'")
    val = int(cur.fetchone()[1])
    cur.close(); conn.close()
    return val

before = get_redo_bytes()
conn = get_connection(); cur = conn.cursor()
for i in range(500):
    cur.execute("INSERT INTO posts (post_id, author_id, content) VALUES (%s, 'U001', 'Test Redo')", (f'R{i}',))
conn.commit(); cur.close(); conn.close()

after = get_redo_bytes()
print(f"Redo bytes generated: {after - before} bytes (approximate)")

# สรุปเชิงวิเคราะห์ (พิมพ์ลงใน Text Cell)

ACID & Locking: การมัดรวมคำสั่งหลายอันไว้ใน Transaction เดียวกัน (Atomicity) ช่วยป้องกันไม่ให้ฐานข้อมูลทำงานสำเร็จแค่ครึ่งเดียว หากเกิด Error ต้องนำคำสั่ง conn.rollback() มาไว้ใน Exception path เพื่อเคลียร์สถานะกลับสู่จุดเริ่มต้นเสมอ

Lost Update vs Atomic UPDATE: ปัญหา Lost Update เกิดจาก Read-Modify-Write ที่ดึงค่ามาประมวลผลก่อนแล้วค่อยเขียนทับ สามารถแก้ไขได้โดยใช้ Atomic UPDATE (เช่น count = count + 1) ซึ่งสั่งให้ InnoDB ล็อกแถวและจัดการอัปเดตให้อัตโนมัติ ทำให้ข้อมูลไม่คลาดเคลื่อน

Isolation & MVCC: การใช้ READ UNCOMMITTED ทำให้เกิด Dirty Read ควรอัปเกรดเป็น READ COMMITTED หรือ REPEATABLE READ เพื่อให้ระบบสร้าง Snapshot ควบคุมการมองเห็นข้อมูล (MVCC) ป้องกันการนำข้อมูลที่ยังไม่ถูก Commit ไปประมวลผล

Recovery & Deadlock: ระบบจะใช้ Redo Log เพื่อนำข้อมูลกลับคืนมาเมื่อเซิร์ฟเวอร์ล่ม และเมื่อ InnoDB ตรวจพบ Deadlock จะทำการ Rollback ฝั่ง Victim ทิ้งทั้งชุดโดยอัตโนมัติ ฝั่งแอปพลิเคชันจึงมีหน้าที่รับมือโดยการใช้ Exponential Backoff และสั่ง Retry ชุดคำสั่งใหม่ทั้งหมด